# Multi-round decoding **without** ancilla reset

In `example_decoder_3d.ipynb` we explicitly reset every Z-ancilla to `Up`
between rounds. On a real device that reset is an extra operation (measure +
conditional-X, or dissipative reset) that costs time and adds noise. We can
skip it entirely by **reusing the same ancilla across rounds and tracking its
state classically**.

## The bookkeeping

Let $a_r$ be the ancilla state at the *start* of round $r$, $p_r$ the true
data-parity for that stabilizer at round $r$, and $b_r$ the bit reported by
the round-$r$ measurement.

The CNOT chain in the syndrome circuit XORs the data parity into the ancilla,
so after the chain the ancilla is in $|a_r \oplus p_r\rangle$ and the
measurement gives $b_r = a_r \oplus p_r$. With **no reset**, the ancilla
state at the start of round $r{+}1$ equals what was just measured:
$a_{r+1} = b_r$. Therefore

$$p_r \;=\; b_r \oplus b_{r-1}, \qquad d_r \;=\; p_r \oplus p_{r-1} \;=\; b_r \oplus b_{r-2}.$$

So the recipe is: keep running each round's measurement, then for each
stabilizer compute the *syndrome* from two consecutive bits, and the
*detector* from two bits **two rounds apart** (with $b_{-1} = b_0 = 0$).

## What changes for measurement errors

In the reset model, a measurement error at round $k$ corrupts exactly one raw
bit $b_k$, lighting detectors at $k$ and $k{+}1$.

In the no-reset model the same wrong bit feeds into our reconstruction
$p_k = b_k \oplus b_{k-1}$ **and** $p_{k+1} = b_{k+1} \oplus b_k$, so two
*syndromes* are corrupted and the resulting detectors light at $k$ and
$k{+}2$ — i.e., **time edges in the 3D matching graph skip a round**.

Everything else (spatial edges, boundary edges, MWPM, Pauli frame) is
unchanged.

In [1]:
using ITensors, ITensorMPS, LinearAlgebra, Plots, Random
Random.seed!(0xC001)
const threshold = 1e-12
gr();

## 1 · Lattice + helpers (compressed from previous notebooks)

In [2]:
const d = 5
data_coords = vec([(x, y) for x in 0:(d-1), y in 0:(d-1)])
z_aux_list  = Tuple{Float64,Float64}[]
x_aux_list  = Tuple{Float64,Float64}[]
for x in 0:(d-2), y in 0:(d-2)
    push!((x + y) % 2 == 0 ? z_aux_list : x_aux_list, (x + 0.5, y + 0.5))
end
for y in 1:2:(d-2);  push!(z_aux_list, (-0.5,    y + 0.5)); end
for y in 0:2:(d-2);  push!(z_aux_list, (d - 0.5, y + 0.5)); end
for x in 0:2:(d-2);  push!(x_aux_list, (x + 0.5, -0.5));    end
for x in 1:2:(d-2);  push!(x_aux_list, (x + 0.5, d - 0.5)); end

all_coords    = vcat(data_coords, z_aux_list, x_aux_list)
sites         = siteinds("S=1/2", length(all_coords))
site_by_coord = Dict(c => sites[i] for (i, c) in enumerate(all_coords))
const Nz      = length(z_aux_list)

data_neighbors_of(ac) =
    [d for d in data_coords if abs(d[1]-ac[1])==0.5 && abs(d[2]-ac[2])==0.5]
P_up(s) = 0.5 * op("Id", s) + op("Sz", s)
P_dn(s) = 0.5 * op("Id", s) - op("Sz", s)

function measure_Z!(psi, aux_site; cutoff = threshold)
    sz = real(inner(psi', apply(op("Sz", aux_site), psi; cutoff)))
    if rand() < 0.5 + sz
        psi = apply(P_up(aux_site), psi; cutoff);  bit = 0
    else
        psi = apply(P_dn(aux_site), psi; cutoff);  bit = 1
    end
    bit, psi / sqrt(real(inner(psi, psi)))
end

function measure_Z_stab(psi, aux_coord; cutoff = threshold)
    aux_site = site_by_coord[aux_coord]
    nbrs = data_neighbors_of(aux_coord)
    order = length(nbrs) == 4 ? [2, 4, 1, 3] : [1, 2]
    for d_coord in nbrs[order]
        psi = apply(op("CNOT", site_by_coord[d_coord], aux_site), psi; cutoff)
    end
    measure_Z!(psi, aux_site; cutoff)
end

# 2D spatial matching-graph bookkeeping (same as before)
z_aux_containing(q) =
    [i for (i, a) in enumerate(z_aux_list)
       if abs(a[1]-q[1])==0.5 && abs(a[2]-q[2])==0.5]
edge_data  = Dict{Tuple{Int,Int}, Tuple{Int,Int}}()
bedge_data = Dict{Int, Tuple{Int,Int}}()
bnd_qubits_of = Dict{Int, Vector{Tuple{Int,Int}}}()
for q in data_coords
    zs = z_aux_containing(q)
    if length(zs) == 2
        a, b = minmax(zs[1], zs[2])
        edge_data[(a, b)] = q
    elseif length(zs) == 1
        bedge_data[zs[1]] = q
        push!(get!(bnd_qubits_of, zs[1], Tuple{Int,Int}[]), q)
    end
end
println("OK — Nz = ", Nz, ", spatial edges = ", length(edge_data),
        ", boundary edges = ", length(bedge_data))

OK — Nz = 12, spatial edges = 15, boundary edges = 6


## 2 · Simulator: same ancilla, no reset, classical tracking

Three things distinguish this from the reset-style simulator:

1. **No call to `reset_aux!`.** The ancilla quantum state simply persists from
   one round to the next.
2. **Track `aux_offset`** — our classical estimate of each ancilla's state at
   the *start* of the current round. Initialised to zero; updated to the
   reported bit after every measurement (note: in the presence of
   measurement errors, this estimate will be temporarily wrong by exactly the
   amount needed to produce the $k, k{+}2$ detector pattern — that's not a
   bug, that's the mechanism).
3. **Report `syn = raw ⊕ aux_offset_before`** as the syndrome for that round,
   so the rest of the pipeline can pretend it's working in the reset world.

In [3]:
function run_rounds_no_reset(R;
        physical_errors = Dict{Int, Vector{Tuple{Int,Int}}}(),
        meas_errors     = Dict{Int, Vector{Int}}())
    psi = MPS(sites, "Up")
    aux_offset = zeros(Int, Nz)
    raw = Vector{Vector{Int}}(undef, R)
    syn = Vector{Vector{Int}}(undef, R)

    for r in 1:R
        # Inject physical errors before round r
        for q in get(physical_errors, r, Tuple{Int,Int}[])
            psi = apply(op("X", site_by_coord[q]), psi; cutoff = threshold)
        end

        round_raw = Int[]
        round_syn = Int[]
        for (i, ac) in enumerate(z_aux_list)
            bit, psi = measure_Z_stab(psi, ac)        # no reset
            push!(round_raw, bit)
            push!(round_syn, bit ⊻ aux_offset[i])     # subtract running offset
            aux_offset[i] = bit                       # advance our estimate
        end

        # Classical measurement errors: flip the recorded bit, leave aux qubit alone
        for i in get(meas_errors, r, Int[])
            round_raw[i] = 1 - round_raw[i]
            # recompute the recorded syndrome and updated offset using the wrong bit
            # (this is exactly what happens in a real device — the experimenter only
            #  sees the noisy classical record)
            round_syn[i] = round_raw[i] ⊻ (aux_offset[i] ⊻ round_raw[i] ⊻ (1 - round_raw[i]))
            # The line above simplifies to: round_syn[i] = 1 - round_syn[i]
            # and aux_offset[i] gets the flipped bit. Make it explicit:
            round_syn[i] = 1 - round_syn[i]
            aux_offset[i] = round_raw[i]
        end

        raw[r] = round_raw
        syn[r] = round_syn
    end

    # Detectors d_r = syn_r XOR syn_{r-1}  (= b_r XOR b_{r-2} in the raw bits)
    detectors = Vector{Vector{Int}}(undef, R)
    prev = zeros(Int, Nz)
    for r in 1:R
        detectors[r] = syn[r] .⊻ prev
        prev = syn[r]
    end

    (; raw, syn, detectors, psi)
end;

## 3 · The 3D matching graph with $r{+}2$ time edges

Identical to the previous notebook except the time edges connect rounds
**two** apart instead of one. We also expose an `edge_kind` classifier so the
decoder can sort each matched edge into spatial / time / boundary.

In [4]:
function build_3d_graph_noreset(R::Int)
    Ntot = R * Nz + 1
    BND  = Ntot
    node_id(i, r) = (r - 1) * Nz + i
    decode_id(u)  = (mod1(u, Nz), div(u - 1, Nz) + 1)

    dist = fill(Inf, Ntot, Ntot)
    nxt  = fill(-1,  Ntot, Ntot)
    for i in 1:Ntot
        dist[i, i] = 0.0; nxt[i, i] = i
    end
    add_edge!(u, v) = begin
        dist[u, v] = 1.0; dist[v, u] = 1.0
        nxt[u, v]  = v;   nxt[v, u]  = u
    end

    # spatial edges (same as reset model)
    for r in 1:R, ((i, j), _) in edge_data
        add_edge!(node_id(i, r), node_id(j, r))
    end
    # time edges: round r <-> round r+2
    for r in 1:(R-2), i in 1:Nz
        add_edge!(node_id(i, r), node_id(i, r + 2))
    end
    # boundary edges (same)
    for r in 1:R, (i, _) in bedge_data
        add_edge!(node_id(i, r), BND)
    end

    for k in 1:Ntot, i in 1:Ntot, j in 1:Ntot
        if dist[i, k] + dist[k, j] < dist[i, j]
            dist[i, j] = dist[i, k] + dist[k, j]
            nxt[i, j]  = nxt[i, k]
        end
    end

    function edge_kind(u, v)
        u, v = minmax(u, v)
        if v == BND
            i, _ = decode_id(u)
            return (:boundary, bedge_data[i])
        end
        iu, ru = decode_id(u)
        iv, rv = decode_id(v)
        if ru == rv
            a, b = minmax(iu, iv)
            return (:spatial, edge_data[(a, b)])
        else
            return (:time, (iu, min(ru, rv)))    # measurement error at the earlier round
        end
    end

    function path_between(a, b)
        nxt[a, b] == -1 && return Int[]
        p = [a]
        while a != b
            a = nxt[a, b]; push!(p, a)
        end
        p
    end

    (; dist, nxt, BND, node_id, decode_id, edge_kind, path_between, R)
end;

## 4 · MWPM + decode (unchanged from previous notebook)

In [5]:
function mwpm(defects::Vector{Int}, dist::Matrix{Float64}, BND::Int)
    isempty(defects) && return Tuple{Int,Int}[], 0.0
    nd    = length(defects)
    nodes = vcat(defects, fill(BND, nd))
    N     = length(nodes)
    W = zeros(N, N)
    for i in 1:N, j in (i+1):N
        if nodes[i] == BND && nodes[j] == BND
            W[i, j] = 0.0
        else
            W[i, j] = dist[nodes[i], nodes[j]]
        end
        W[j, i] = W[i, j]
    end
    function rec(rem)
        isempty(rem) && return 0.0, Tuple{Int,Int}[]
        first  = rem[1]; best_c = Inf; best_m = Tuple{Int,Int}[]
        for k in 2:length(rem)
            c = W[first, rem[k]]
            isfinite(c) || continue
            sub_rem = vcat(rem[2:k-1], rem[k+1:end])
            sc, sm  = rec(sub_rem)
            if c + sc < best_c
                best_c = c + sc; best_m = vcat([(first, rem[k])], sm)
            end
        end
        best_c, best_m
    end
    cost, m_idx = rec(collect(1:N))
    pairs = [(nodes[i], nodes[j]) for (i, j) in m_idx
             if !(nodes[i] == BND && nodes[j] == BND)]
    pairs, cost
end

function decode(detectors::Vector{Vector{Int}}, g)
    R = g.R
    lit = Int[]
    for r in 1:R, i in 1:Nz
        if detectors[r][i] == 1
            push!(lit, g.node_id(i, r))
        end
    end
    pairs, cost = mwpm(lit, g.dist, g.BND)
    x_qubits = Tuple{Int,Int}[]; meas_err = Tuple{Int,Int}[]
    for (a, b) in pairs
        p = g.path_between(a, b)
        for k in 1:(length(p)-1)
            kind = g.edge_kind(p[k], p[k+1])
            if kind[1] === :time
                push!(meas_err, kind[2])
            else
                push!(x_qubits, kind[2])
            end
        end
    end
    counts = Dict{Tuple{Int,Int},Int}()
    for q in x_qubits; counts[q] = get(counts, q, 0) + 1; end
    frame_update = [q for (q, c) in counts if isodd(c)]
    (; lit, pairs, frame_update, meas_err, cost)
end;

## 5 · Visualisations

Same per-round panels and stacked-3D plot as before — only thing that
changes visually is that time edges in the 3D plot now skip a round.

In [6]:
outward_point(zaux_idx, data_q; reach = 0.7) = begin
    a = z_aux_list[zaux_idx]
    dx, dy = data_q[1]-a[1], data_q[2]-a[2]
    (a[1] + (1+reach) * dx * 2, a[2] + (1+reach) * dy * 2)
end

function plot_round_panel(r, detectors, pairs, g; meas_err = Tuple{Int,Int}[])
    p = plot(; aspect_ratio = :equal, legend = false, framestyle = :box,
              xlims = (-1.6, d + 0.6), ylims = (-1.6, d + 0.6),
              size = (380, 400), title = "round $r")
    scatter!(p, [q[1] for q in data_coords], [q[2] for q in data_coords];
             color = :grey75, ms = 3)
    for ((i, j), _) in edge_data
        a, b = z_aux_list[i], z_aux_list[j]
        plot!(p, [a[1], b[1]], [a[2], b[2]]; color = :black, lw = 0.8, alpha = 0.35)
    end
    for (i, qs) in bnd_qubits_of, q in qs
        a = z_aux_list[i]; o = outward_point(i, q)
        plot!(p, [a[1], o[1]], [a[2], o[2]];
              color = :grey50, lw = 0.8, alpha = 0.5, linestyle = :dash)
    end
    for (u, v) in pairs
        path = g.path_between(u, v)
        for k in 1:(length(path) - 1)
            kind = g.edge_kind(path[k], path[k+1])
            (kind[1] === :spatial || kind[1] === :boundary) || continue
            iu, ru = g.decode_id(path[k])
            iv = path[k+1] == g.BND ? nothing : g.decode_id(path[k+1])[1]
            rv = path[k+1] == g.BND ? ru     : g.decode_id(path[k+1])[2]
            ru == r == rv || continue
            if path[k+1] == g.BND
                a = z_aux_list[iu]; o = outward_point(iu, kind[2])
                plot!(p, [a[1], o[1]], [a[2], o[2]];
                      color = :orange, lw = 3, alpha = 0.85)
            else
                a = z_aux_list[iu]; b = z_aux_list[iv]
                plot!(p, [a[1], b[1]], [a[2], b[2]];
                      color = :orange, lw = 3, alpha = 0.85)
            end
        end
    end
    scatter!(p, [a[1] for a in z_aux_list], [a[2] for a in z_aux_list];
             color = :cyan, ms = 9, marker = :diamond,
             markerstrokewidth = 1, markerstrokecolor = :black)
    lit = [z_aux_list[i] for (i, b) in enumerate(detectors[r]) if b == 1]
    if !isempty(lit)
        scatter!(p, [a[1] for a in lit], [a[2] for a in lit];
                 color = :red, ms = 12, marker = :diamond,
                 markerstrokewidth = 2, markerstrokecolor = :black)
    end
    for (stab, rr) in meas_err
        if rr == r
            a = z_aux_list[stab]
            scatter!(p, [a[1]], [a[2]];
                     color = :blue, marker = :circle, ms = 15, alpha = 0,
                     markerstrokewidth = 2.5, markerstrokecolor = :blue)
            annotate!(p, a[1] + 0.35, a[2] + 0.35, text("M", 9, :blue, :bold))
        end
    end
    p
end

plot_all_rounds(detectors, pairs, g; meas_err = Tuple{Int,Int}[]) =
    plot([plot_round_panel(r, detectors, pairs, g; meas_err) for r in 1:g.R]...;
         layout = (1, g.R), size = (380 * g.R, 400))

function plot_3d(detectors, pairs, g)
    R = g.R
    p = plot(; size = (820, 620), xlabel = "x", ylabel = "y", zlabel = "round",
              camera = (35, 22), legend = false,
              title = "3D matching graph (no-reset model, r↔r+2 time edges)")
    for r in 1:R
        scatter!(p, [a[1] for a in z_aux_list], [a[2] for a in z_aux_list],
                 fill(float(r), Nz); color = :cyan, ms = 5, marker = :diamond,
                 markerstrokewidth = 0.4, markerstrokecolor = :black)
    end
    for r in 1:R, ((i, j), _) in edge_data
        a, b = z_aux_list[i], z_aux_list[j]
        plot!(p, [a[1], b[1]], [a[2], b[2]], [float(r), float(r)];
              color = :grey60, lw = 0.6, alpha = 0.35)
    end
    for r in 1:(R-2), i in 1:Nz
        a = z_aux_list[i]
        plot!(p, [a[1], a[1]], [a[2], a[2]], [float(r), float(r + 2)];
              color = :blue, lw = 0.6, alpha = 0.35)
    end
    for r in 1:R, i in 1:Nz
        if detectors[r][i] == 1
            a = z_aux_list[i]
            scatter!(p, [a[1]], [a[2]], [float(r)];
                     color = :red, ms = 8, marker = :diamond,
                     markerstrokewidth = 1, markerstrokecolor = :black)
        end
    end
    for (u, v) in pairs
        path = g.path_between(u, v)
        for k in 1:(length(path) - 1)
            a, b = path[k], path[k+1]
            if a == g.BND || b == g.BND
                sb = a == g.BND ? b : a
                i, r = g.decode_id(sb)
                kind = g.edge_kind(a, b)
                aa = z_aux_list[i]; o = outward_point(i, kind[2])
                plot!(p, [aa[1], o[1]], [aa[2], o[2]], [float(r), float(r)];
                      color = :orange, lw = 3, alpha = 0.85)
            else
                ia, ra = g.decode_id(a); ib, rb = g.decode_id(b)
                aa = z_aux_list[ia]; bb = z_aux_list[ib]
                plot!(p, [aa[1], bb[1]], [aa[2], bb[2]], [float(ra), float(rb)];
                      color = :orange, lw = 3, alpha = 0.85)
            end
        end
    end
    p
end

function print_detector_matrix(detectors)
    R = length(detectors)
    println("       ", join(["r=$r" for r in 1:R], "   "))
    for i in 1:Nz
        row = [detectors[r][i] for r in 1:R]
        marker = any(==(1), row) ? " <-" : ""
        println(" stab ", lpad(i, 2), "  ",
                join([b == 1 ? " 1 " : " . " for b in row], "  "), marker)
    end
end

function summarise(result, decoded)
    print_detector_matrix(result.detectors)
    println()
    println("lit detector nodes: ", decoded.lit)
    println("matched pairs:      ", decoded.pairs)
    println("MWPM cost:          ", decoded.cost)
    println("frame update (X q): ", decoded.frame_update)
    println("inferred meas errs: ", decoded.meas_err)
end;

## 6 · Build the matching graph for $R = 4$ rounds

I'm using $R=4$ because in the no-reset model, a measurement error at round
$k$ lights detectors at $k$ and $k{+}2$. To keep both endpoints inside the
experiment for a mid-round measurement error, we need at least 4 rounds.

In [7]:
const R = 4
const g = build_3d_graph_noreset(R)
println("3D-graph nodes: ", R * Nz + 1)

3D-graph nodes: 49


## Demo 1 · No errors (baseline)

In [8]:
let
    result = run_rounds_no_reset(R)
    decoded = decode(result.detectors, g)
    summarise(result, decoded)
    @assert all(all(==(0), s) for s in result.detectors)
    @assert isempty(decoded.frame_update); @assert isempty(decoded.meas_err)
end

┌ Warning: Calling `inner(x::MPS, A::MPO, y::MPS)` where the site indices of the `MPS`
│ `x` and the `MPS` resulting from contracting `MPO` `A` with `MPS` `y` don't
│ match is deprecated as of ITensors v0.3 and will result in an error in ITensors
│ v0.4. The most common cause of this is something like the following:
│ 
│ ```julia
│ s = siteinds("S=1/2")
│ psi = random_mps(s)
│ H = MPO(s, "Id")
│ inner(psi, H, psi)
│ ```
│ 
│ `psi` has the Index structure `-s-(psi)` and `H` has the Index structure
│ `-s'-(H)-s-`, so the Index structure of would be `(dag(psi)-s- -s'-(H)-s-(psi)`
│  unless the prime levels were fixed. Previously we tried fixing the prime level
│   in situations like this, but we will no longer be doing that going forward.
│ 
│ There are a few ways to fix this. You can simply change:
│ 
│ ```julia
│ inner(psi, H, psi)
│ ```
│ 
│ to:
│ 
│ ```julia
│ inner(psi', H, psi)
│ ```
│ 
│ in which case the Index structure will be `(dag(psi)-s'-(H)-s-(psi)`.
│ 
│ Alternatively, you c

       r=1   r=2   r=3   r=4
 stab  1   .    .    .    . 
 stab  2   .    .    .    . 
 stab  3   .    .    .    . 
 stab  4   .    .    .    . 
 stab  5   .    .    .    . 
 stab  6   .    .    .    . 
 stab  7   .    .    .    . 
 stab  8   .    .    .    . 
 stab  9   .    .    .    . 
 stab 10   .    .    .    . 
 stab 11   .    .    .    . 
 stab 12   .    .    .    . 

lit detector nodes: Int64[]
matched pairs:      Tuple{Int64, Int64}[]
MWPM cost:          0.0
frame update (X q): Tuple{Int64, Int64}[]
inferred meas errs: Tuple{Int64, Int64}[]


## Demo 2 · Physical X error before round 2

Identical to its reset-model counterpart: X on $(2,2)$ before round 2 lights
the two adjacent Z-stabilizers **once** at round 2 — and crucially does *not*
re-fire in round 4. The cumulative-XOR ancilla state has the same parity at
rounds $r$ and $r{+}2$ once the error has persisted for that long, so
$b_r \oplus b_{r-2} = 0$ from round 4 onward.

In [ ]:
let
    result = run_rounds_no_reset(R; physical_errors = Dict(2 => [(2, 2)]))
    decoded = decode(result.detectors, g)
    summarise(result, decoded)
    @assert Set(decoded.frame_update) == Set([(2, 2)])
    @assert isempty(decoded.meas_err)
    display(plot_all_rounds(result.detectors, decoded.pairs, g))
    display(plot_3d(result.detectors, decoded.pairs, g))
end

## Demo 3 · One measurement error at round 2

This is where the difference from the reset model becomes visible. A single
measurement error at stab 3, round 2 lights detectors at **rounds 2 *and*
4**, paired by a single $r{+}2$ time edge.

In [ ]:
let
    target_stab = 3
    result = run_rounds_no_reset(R; meas_errors = Dict(2 => [target_stab]))
    decoded = decode(result.detectors, g)
    summarise(result, decoded)
    @assert isempty(decoded.frame_update)
    @assert Set(decoded.meas_err) == Set([(target_stab, 2)])
    display(plot_all_rounds(result.detectors, decoded.pairs, g;
                            meas_err = decoded.meas_err))
    display(plot_3d(result.detectors, decoded.pairs, g))
end

## Demo 4 · Combined: physical + measurement error

X on $(1, 2)$ before round 2 *and* a measurement error at stab 8 in round 2.
MWPM should commit:

- one spatial pair (cost 1) for the physical error at round 2, and
- one $r{+}2$ time pair (cost 1) at stab 8 between rounds 2 and 4

for total cost 2. The frame update is just $(1, 2)$; the inferred
measurement error is `(8, 2)`.

In [ ]:
let
    target_stab = 8
    result = run_rounds_no_reset(R;
        physical_errors = Dict(2 => [(1, 2)]),
        meas_errors     = Dict(2 => [target_stab]))
    decoded = decode(result.detectors, g)
    summarise(result, decoded)
    @assert Set(decoded.frame_update) == Set([(1, 2)])
    @assert Set(decoded.meas_err)     == Set([(target_stab, 2)])
    display(plot_all_rounds(result.detectors, decoded.pairs, g;
                            meas_err = decoded.meas_err))
    display(plot_3d(result.detectors, decoded.pairs, g))
end

## 7 · Final logical readout

After all rounds finish, destructively measure every data qubit and combine
that with the Pauli frame to read out the logical observable.

**Choosing $Z_L$ and $X_L$.** In our staggered layout, the X-boundaries are
*top* and *bottom* (top/bottom 2-body stabilizers are X-type) and the
Z-boundaries are *left* and *right*. A logical operator must terminate at
the matching pair of boundaries:

- $Z_L$ = a chain of $Z$'s between Z-boundaries → a **horizontal** chain;
  we'll use the bottom row $y=0$.
- $X_L$ = a chain of $X$'s between X-boundaries → a **vertical** chain;
  we'll use the left column $x=0$.

These anti-commute on the shared data qubit $(0, 0)$ ✓ and commute with every
stabilizer ✓.

**Frame contribution.**

- $Z_L$ readout: $X$ corrections in the frame *flip* the recorded $Z$-eigenvalue
  on any data qubit in $Z_L$'s support, so
  $\textsf{logical } Z_L = (\text{raw } Z_L\text{ parity}) \oplus
   (\text{parity of frame on }Z_L\text{ support})$.
- $X_L$ readout: $X$ corrections commute with $X$, so they leave $X$-basis
  eigenvalues alone. The frame doesn't contribute and
  $\textsf{logical } X_L = (\text{raw } X_L\text{ parity})$.

**Why we can't read both in one shot.** $Z_L$ and $X_L$ anti-commute. To
measure $X_L$ we apply Hadamards to every data qubit, which destroys $Z_L$.
The two readouts have to live in separate simulator runs (the same as in a
real lab — you pick a basis).

In [9]:
const ZL_support = [(x, 0)        for x in 0:(d-1)]   # bottom row, Z chain
const XL_support = [(0, y)        for y in 0:(d-1)]   # left column, X chain

# Destructive single-qubit Z measurement on every data qubit.
# Returns Dict{coord -> bit} with bit = 0 (Up = |0>) / 1 (Dn = |1>).
function measure_data_Z(psi_in; cutoff = threshold)
    psi = copy(psi_in)
    bits = Dict{Tuple{Int,Int}, Int}()
    for q in data_coords
        site = site_by_coord[q]
        sz = real(inner(psi', apply(op("Sz", site), psi; cutoff)))
        if rand() < 0.5 + sz
            psi = apply(P_up(site), psi; cutoff); bits[q] = 0
        else
            psi = apply(P_dn(site), psi; cutoff); bits[q] = 1
        end
        psi = psi / sqrt(real(inner(psi, psi)))
    end
    bits
end

# Destructive X-basis readout: H on each data qubit first, then Z-measure.
function measure_data_X(psi_in; cutoff = threshold)
    psi = copy(psi_in)
    for q in data_coords
        psi = apply(op("H", site_by_coord[q]), psi; cutoff)
    end
    measure_data_Z(psi; cutoff)
end

# Combine raw readout with the frame.
function logical_Z(bits, frame_update; support = ZL_support)
    raw          = mod(sum(bits[q] for q in support), 2)
    frame_parity = mod(count(q -> q in support, frame_update), 2)
    raw ⊻ frame_parity
end
logical_X(bits; support = XL_support) =
    mod(sum(bits[q] for q in support), 2)   # frame doesn't affect X-basis

logical_X (generic function with 1 method)

### Demo 5 · $Z_L$ readout on the no-error baseline

Just runs four rounds with no errors and reads $Z_L$. The state is still
$|0\dots 0\rangle$ → every data bit is 0 → raw $Z_L = 0$, frame empty,
logical $Z_L = 0$.

In [10]:
let
    result   = run_rounds_no_reset(R)
    decoded  = decode(result.detectors, g)
    bits     = measure_data_Z(result.psi)
    println("data bits along Z_L: ", [bits[q] for q in ZL_support])
    println("frame ∩ Z_L:         ", [q for q in decoded.frame_update if q in ZL_support])
    logZ = logical_Z(bits, decoded.frame_update)
    println("logical Z_L = ", logZ, "   (expect 0)")
    @assert logZ == 0
end

data bits along Z_L: [0, 0, 0, 0, 0]
frame ∩ Z_L:         Tuple{Int64, Int64}[]
logical Z_L = 0   (expect 0)


### Demo 6 · Decoder rescues a $Z_L$-touching error

X on data $(2, 0)$ before round 2. That qubit lies in only one Z-stabilizer
($(2.5, 0.5)$), so a single boundary defect fires at round 2. MWPM matches
it to the boundary; the boundary edge is labelled by *another* data qubit on
the bottom row (`(3, 0)` here). So:

- Raw $Z_L$ parity flips by 1 (data $(2,0)$ is now $|1\rangle$).
- Frame parity on $Z_L$ also flips by 1 (the correction is X on $(3,0)$,
  which lies on $Z_L$).

The two flips cancel: logical $Z_L = 0$. The decoder's correction differs
from the true error by an X-type 2-body stabilizer at $(2.5, -0.5)$, so the
residual is a stabilizer — exactly the case where decoding is a *successful*
correction modulo stabilizers.

In [11]:
let
    result   = run_rounds_no_reset(R; physical_errors = Dict(2 => [(2, 0)]))
    decoded  = decode(result.detectors, g)
    bits     = measure_data_Z(result.psi)
    println("data bits along Z_L: ", [bits[q] for q in ZL_support])
    println("frame_update:        ", decoded.frame_update)
    println("frame ∩ Z_L:         ", [q for q in decoded.frame_update if q in ZL_support])
    raw_parity   = mod(sum(bits[q] for q in ZL_support), 2)
    frame_parity = mod(count(q -> q in ZL_support, decoded.frame_update), 2)
    println("raw Z_L = ", raw_parity,
            ",  frame parity on Z_L = ", frame_parity,
            ",  logical Z_L = ", raw_parity ⊻ frame_parity)
    @assert (raw_parity ⊻ frame_parity) == 0
end

data bits along Z_L: [0, 0, 1, 0, 0]
frame_update:        [(3, 0)]
frame ∩ Z_L:         [(3, 0)]
raw Z_L = 1,  frame parity on Z_L = 1,  logical Z_L = 0


### Demo 7 · Injecting $X_L$ → undetected logical error

Apply the full chain $X_{(0,0)} X_{(0,1)} X_{(0,2)} X_{(0,3)} X_{(0,4)}$
(every qubit in $X_L$'s support) before round 1. This is a logical $X_L$.

$X_L$ commutes with every Z-stabilizer by construction, so **all syndromes
stay at zero**. The decoder sees nothing wrong, returns an empty frame, and
applies no correction. But the state has actually flipped: data $(0, 0)$ is
now $|1\rangle$ and the other $Z_L$-support qubits are still $|0\rangle$, so
raw $Z_L = 1$. With an empty frame, logical $Z_L = 1$ — an undetected
logical error.

This is the canonical failure mode of MWPM: errors that look like a logical
operator are indistinguishable from "no error" given only the syndrome.

In [12]:
let
    result   = run_rounds_no_reset(R; physical_errors = Dict(1 => XL_support))
    decoded  = decode(result.detectors, g)
    println("any defects? ", any(any(==(1), s) for s in result.detectors))
    println("frame:        ", decoded.frame_update)
    bits     = measure_data_Z(result.psi)
    println("data bits along Z_L: ", [bits[q] for q in ZL_support])
    logZ = logical_Z(bits, decoded.frame_update)
    println("logical Z_L = ", logZ, "   (EXPECT 1: undetected logical error)")
    @assert logZ == 1
end

any defects? false
frame:        Tuple{Int64, Int64}[]
data bits along Z_L: [1, 0, 0, 0, 0]
logical Z_L = 1   (EXPECT 1: undetected logical error)


### Demo 8 · $X_L$ readout on $|0\rangle_L$ is random

$|0\rangle_L$ is *not* an $X_L$ eigenstate (it's a superposition of
$|+\rangle_L$ and $|-\rangle_L$ with equal amplitude). So measuring $X_L$
must give 50/50.

We run the no-error baseline 30 times, each ending with an X-basis
destructive measurement, and tally how often $X_L$ comes out 0 vs 1.
Expectation: $\sim 15$ of each.

In [13]:
let
    counts = [0, 0]
    for _ in 1:30
        result = run_rounds_no_reset(R)
        bits   = measure_data_X(result.psi)
        logX   = logical_X(bits)
        counts[logX + 1] += 1
    end
    println("X_L outcomes over 30 trials:  0 → ", counts[1],
            "  1 → ", counts[2])
    println("(expect ~15 / ~15)")
end

X_L outcomes over 30 trials:  0 → 16  1 → 14
(expect ~15 / ~15)
